In [4]:
import os
import pandas as pd
from sklearn.model_selection import StratifiedKFold
import numpy as np

In [5]:
def make_stratified_splits(
    csv_path,
    n_splits=10,
    output_dir="stratified_splits",
):
    """
    Fully deterministic split strategy for GTZAN based on within-genre track index.

    For split k (0-indexed):
      - test  : tracks where track_idx is in [k*10, k*10+9]         (10 per genre)
      - val   : tracks where track_idx is in [(k+1)%10*10, ...]     (10 per genre)
      - train : all remaining tracks                                 (80 per genre)

    This guarantees:
      - Every sample is test   in exactly 1 fold
      - Every sample is val    in exactly 1 fold
      - Every sample is train  in exactly 8 folds
      - No randomness — splits are fully reproducible without a random seed
    """
    df = pd.read_csv(csv_path).reset_index(drop=True)

    # extract within-genre track index from filename e.g. "blues.00013.wav" → 13
    df["track_idx"] = (
        df["filename"]
        .str.extract(r"\.(\d+)\.wav$")
        .astype(int)
    )
    
    rng = np.random.default_rng(seed=42)
    
    def shuffle_within_genre(group):
        shuffled_idx = rng.permutation(len(group))
        group = group.copy()
        group["track_idx"] = shuffled_idx
        return group

    df = df.groupby("label", group_keys=False).apply(shuffle_within_genre)


    # assign each track to a bucket 0-9 based on its index within the genre
    bucket_size = 100 // n_splits   # = 10
    df["bucket"] = df["track_idx"] // bucket_size

    os.makedirs(output_dir, exist_ok=True)

    col_names = [f"fold_{i}" for i in range(n_splits)]
    summary = pd.DataFrame("—", index=df.index, columns=col_names)
    folds = []

    for fold_idx in range(n_splits):
        col = f"fold_{fold_idx}"

        test_bucket = fold_idx
        val_bucket  = (fold_idx + 1) % n_splits   # wrap around to 0

        test_mask  = df["bucket"] == test_bucket
        val_mask   = df["bucket"] == val_bucket
        train_mask = ~test_mask & ~val_mask

        test_df  = df[test_mask].copy()
        val_df   = df[val_mask].copy()
        train_df = df[train_mask].copy()

        summary.loc[test_df.index,  col] = "test"
        summary.loc[val_df.index,   col] = "val"
        summary.loc[train_df.index, col] = "train"

        # drop helper columns before saving
        drop_cols = ["track_idx", "bucket"]
        train_out = train_df.drop(columns=drop_cols)
        val_out   = val_df.drop(columns=drop_cols)
        test_out  = test_df.drop(columns=drop_cols)

        fold_dir = os.path.join(output_dir, f"fold_{fold_idx}")
        os.makedirs(fold_dir, exist_ok=True)
        train_out.to_csv(os.path.join(fold_dir, "train.csv"), index=False)
        val_out.to_csv(  os.path.join(fold_dir, "val.csv"),   index=False)
        test_out.to_csv( os.path.join(fold_dir, "test.csv"),  index=False)

        folds.append({"train": train_out, "val": val_out, "test": test_out})

        print(
            f"Fold {fold_idx} — "
            f"test: bucket {test_bucket} (idx {test_bucket*bucket_size:02d}-{test_bucket*bucket_size+bucket_size-1:02d})  "
            f"val: bucket {val_bucket} (idx {val_bucket*bucket_size:02d}-{val_bucket*bucket_size+bucket_size-1:02d})  "
            f"train={len(train_out)}  val={len(val_out)}  test={len(test_out)}"
        )

    role_cols = summary[col_names]
    summary = pd.concat([df[["filepath", "filename", "label"]], summary], axis=1)
    summary["n_train"] = (role_cols == "train").sum(axis=1)
    summary["n_val"]   = (role_cols == "val").sum(axis=1)
    summary["n_test"]  = (role_cols == "test").sum(axis=1)

    summary_path = os.path.join(output_dir, "fold_summary.csv")
    summary.to_csv(summary_path, index=False)
    print(f"\nSummary saved to {summary_path}")
    print("\nRole counts across all folds:")
    print(summary[["n_train", "n_val", "n_test"]].describe().loc[["min", "max"]])

    return folds, summary


In [6]:
DATA_ROOT = "./gtzan"
all_data_path = "metadata_all.csv"

folds, summary = make_stratified_splits(all_data_path, output_dir="stratified_splits_shuffled")

/tmp/ipykernel_292311/2332023520.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("label", group_keys=False).apply(shuffle_within_genre)


Fold 0 — test: bucket 0 (idx 00-09)  val: bucket 1 (idx 10-19)  train=800  val=100  test=100
Fold 1 — test: bucket 1 (idx 10-19)  val: bucket 2 (idx 20-29)  train=800  val=100  test=100
Fold 2 — test: bucket 2 (idx 20-29)  val: bucket 3 (idx 30-39)  train=800  val=100  test=100
Fold 3 — test: bucket 3 (idx 30-39)  val: bucket 4 (idx 40-49)  train=800  val=100  test=100
Fold 4 — test: bucket 4 (idx 40-49)  val: bucket 5 (idx 50-59)  train=800  val=100  test=100
Fold 5 — test: bucket 5 (idx 50-59)  val: bucket 6 (idx 60-69)  train=800  val=100  test=100
Fold 6 — test: bucket 6 (idx 60-69)  val: bucket 7 (idx 70-79)  train=800  val=100  test=100
Fold 7 — test: bucket 7 (idx 70-79)  val: bucket 8 (idx 80-89)  train=800  val=100  test=100
Fold 8 — test: bucket 8 (idx 80-89)  val: bucket 9 (idx 90-99)  train=800  val=100  test=100
Fold 9 — test: bucket 9 (idx 90-99)  val: bucket 0 (idx 00-09)  train=800  val=100  test=100

Summary saved to stratified_splits_shuffled/fold_summary.csv

Role co

In [11]:
summary.to_csv("stratified_suffled_meta.csv")